# Managed Vector Databases: Pinecone & Feature Comparison
This notebook introduces **Pinecone**, a fully managed cloud-native vector database designed for high-scale AI and RAG applications. It includes a comprehensive feature comparison across **FAISS**, **Chroma**, and **Pinecone**, followed by a step-by-step implementation of serverless index creation, document upsertion, metadata filtering, threshold retrieval, and deletion.

---

#### FAISS vs Chroma vs Pinecone

| Feature | **FAISS** | **Chroma** | **Pinecone** |
|---------|-----------|------------|--------------|
| **Product Type** | Vector search library | Vector database | Managed vector database |
| **Runs Locally** | ✅ Yes | ✅ Yes | ❌ No (cloud-first service) |
| **Managed Cloud** | ❌ No | ✅ Yes | ✅ Yes (primary deployment model) |
| **Vector Index** | Directly controlled | Database-managed | Fully managed |
| **Documents** | LangChain or external document store | Native collection records | Stored as metadata/records (or via integrations) |
| **Metadata** | External wrapper | Native | Native |
| **Metadata Filtering** | ❌ Not native | ✅ Native | ✅ Native |
| **CRUD Operations** | Limited (index-dependent) | ✅ Native | ✅ Native |
| **Persistence** | Manual save/load | Automatic client/server persistence | Fully managed |
| **Collections** | ❌ No native collection primitive | ✅ Native collections | Indexes and namespaces |
| **Scaling** | User-managed | Local, server, or cloud deployment | Automatically managed infrastructure |
| **Server API** | Build your own | Available | Built-in |
| **GPU / Index Tuning** | Full control | Abstracted | Fully abstracted |
| **Cost** | Infrastructure cost only | Free locally, paid cloud options | Usage-based managed cloud pricing |
| **Best For** | Research, custom ANN algorithms, local development | Local RAG applications and flexible deployments | Production-scale cloud retrieval systems |

## 1. Setup & Library Imports
Import essential modules from `pinecone`, `langchain_pinecone`, `langchain_google_genai`, and `langchain_core`.

In [1]:
# Load required libraries for Pinecone, Google Embeddings, and LangChain Document handling
import os
import time
from uuid import uuid4
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_core.documents import Document

d:\Coding\Full-Stack-GenAI-AgenticAI-Bootcamp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Environment Variables Validation
Load keys from the `.env` file and verify that both `GOOGLE_API_KEY` and `PINECONE_API_KEY` are configured.

In [2]:
# Load environment variables and validate API keys

load_dotenv()

google_api_key = os.getenv("GOOGLE_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")

if not google_api_key:
    raise ValueError("GOOGLE_API_KEY is missing")

if not pinecone_api_key:
    raise ValueError("PINECONE_API_KEY is missing")

## 3. Initialize Embedding Model & Check Dimension
Instantiate `GoogleGenerativeAIEmbeddings` and check vector dimensionality (768 for `gemini-embedding-001`).

In [3]:
# Initialize Google Generative AI embedding model and check output dimension
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embedding_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
dimension = len(
    embedding_model.embed_query("dimension check")
)
print("Embedding dimension:", dimension)

Embedding dimension: 3072


## 4. Initialize Pinecone Client & Define Index Name
Connect to the Pinecone cloud service using the SDK client.

In [4]:
# Initialize Pinecone client with API key
pc = Pinecone(
    api_key=pinecone_api_key
)

In [5]:
# Define target Pinecone index name
index_name = "langchain-llama-index"

## 5. Create Serverless Index
Provision a serverless index on AWS (`us-east-1`) with Cosine similarity metric if it does not already exist.

In [6]:
# Create serverless Pinecone index if it doesn't already exist
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        ),
    )

## 6. Poll Index Status Until Ready
Check the provisioning status of the Pinecone cloud index until it is ready for operations.

In [9]:
# Query single-shot index readiness status
description = pc.describe_index(index_name)
description.status["ready"]

True

In [10]:
# Poll index status in a loop until fully ready
while True:
    description = pc.describe_index(index_name)
    if description.status["ready"]:
        break
    time.sleep(2)

## 7. Connect to Index & Wrap in LangChain
Attach the Pinecone index object to LangChain's `PineconeVectorStore` using a custom namespace.

In [11]:
# Connect to the created Pinecone index
index = pc.Index(index_name)

In [13]:
# Initialize LangChain PineconeVectorStore with specified namespace
vector_store = PineconeVectorStore(
    index=index,
    embedding=embedding_model,
    namespace="demo-documents"
)

## 8. Create Sample Documents & Unique IDs
Instantiate structured LangChain `Document` objects with metadata and assign unique UUIDs.

In [14]:
# Define sample Document objects with metadata tags and generate UUIDs
documents = [
    Document(
        page_content=(
            "I had chocolate chip pancakes and "
            "scrambled eggs for breakfast this morning."
        ),
        metadata={"source": "tweet"},
    ),
    Document(
        page_content=(
            "The weather forecast for tomorrow is cloudy "
            "and overcast, with a high of 62 degrees."
        ),
        metadata={"source": "news"},
    ),
    Document(
        page_content=(
            "Building an exciting new project with "
            "LangChain - come check it out!"
        ),
        metadata={"source": "tweet"},
    ),
    Document(
        page_content=(
            "Robbers broke into the city bank and "
            "stole $1 million in cash."
        ),
        metadata={"source": "news"},
    ),
    Document(
        page_content=(
            "LangGraph is the best framework for building "
            "stateful, agentic applications!"
        ),
        metadata={"source": "tweet"},
    ),
]

ids = [
    str(uuid4())
    for _ in documents
]

## 9. Upsert Documents to Pinecone Cloud
Upload the document embeddings, content, and metadata to the serverless Pinecone index.

In [15]:
# Upsert documents and IDs into Pinecone vector store
inserted_ids = vector_store.add_documents(
    documents=documents,
    ids=ids
)

print("Inserted IDs:", inserted_ids)

Inserted IDs: ['0b849e61-a3f6-441f-9d33-f99dc3378d12', '7acb4902-4337-4bf1-bdb6-7446f69475cf', '0e9658e8-ad5f-4d6b-bf77-0eab591fc2c6', '8818687a-4e1c-41a8-ae1a-78b9921876a9', 'fee164a4-0bcb-49f9-8ebe-1b37e88ca933']


## 10. Similarity Search with Metadata Filtering
Perform semantic similarity search on Pinecone with metadata filtering (e.g. `source == 'tweet'`).

In [16]:
# Search top 2 similar documents matching metadata filter 'source: tweet'

results = vector_store.similarity_search(
    query=(
        "LangChain provides abstractions "
        "for working with LLMs"
    ),
    k=2,
    filter={
        "source": "tweet"
    }
)

for result in results:
    print(result.page_content)
    print(result.metadata)

Building an exciting new project with LangChain - come check it out!
{'source': 'tweet'}
LangGraph is the best framework for building stateful, agentic applications!
{'source': 'tweet'}


## 11. Score Threshold Retriever Configuration
Create a retriever using `similarity_score_threshold` combined with metadata filters.

In [17]:
# Configure score threshold retriever and test query
retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 1,
        "score_threshold": 0.4,
        "filter": {
            "source": "news"
        }
    }
)

retrieved_docs = retriever.invoke(
    "Stealing money from a bank is a crime"
)

print(retrieved_docs)

[Document(id='8818687a-4e1c-41a8-ae1a-78b9921876a9', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]


## 12. Delete Document Records by ID
Demonstrate document deletion by vector ID from the Pinecone index.

In [18]:
# Delete specific vector record by ID
vector_store.delete(
    ids=[ids[-1]]
)